In [1]:
# 08d Re-validate GLM v2 on the corrected target (excess on every peril)
# Purpose: (1) benchmark for the LightGBM challenger: same target, folds and metrics;
# (2) one-year-ahead A/E per line: how much a model trained on past years under-predicts the next year.
import sys, time
sys.path.append("..")
from pathlib import Path
import numpy as np
import pandas as pd
from src.target import LINES, load_policies, load_claims, build_model_target
from src.features import build_features
from src.glm import BandedTweedieGLM
from src.evaluate import evaluate
from src.diagnostics import FOLDS

CONFIG = {
    "motor":     dict(alpha=0.5, n_bands=10, drop=None, balance=True),
    "buildings": dict(alpha=0.5, n_bands=20, drop=None, balance=True),
    "contents":  dict(alpha=0.3, n_bands=20, drop=None, balance=True),
}
NON_FEATURES = ("row_id", "year", "exposure_clean", "target", "weight")

claims = load_claims()
data, rows, valid_preds = {}, [], []
for line in LINES:
    p = load_policies(line, "train")
    mt, comp, loading = build_model_target(claims, p, line)          # default: excess on every peril
    df = mt.merge(build_features(line, "train", policies=p), on="row_id", how="left", validate="one_to_one")
    data[line] = {"df": df, "loading": loading}
    feats = [c for c in df.columns if c not in NON_FEATURES]
    for vy, tys in FOLDS:
        t0 = time.time()
        tr, va = df[df.year.isin(tys)], df[df.year == vy]
        glm = BandedTweedieGLM(**CONFIG[line]).fit(tr[feats], tr.target / tr.weight, tr.weight)
        pred = glm.predict_rate(va[feats]) * va.weight.to_numpy()
        r = evaluate(va.target, pred, va.weight, show=False)
        lf = r["lift"]["A/E"]
        rows.append({"line": line, "year": vy, "k": glm.k_, "D2": r["D2"], "A/E": r["A/E"],
                     "low": lf.iloc[0], "top": lf.iloc[-1]})
        valid_preds.append(pd.DataFrame({"line": line, "year": vy, "row_id": va.row_id.values,
                                         "actual": va.target.values, "pred": pred, "weight": va.weight.values}))
        print(f"{line} {vy}: {time.time() - t0:.0f}s")

res = pd.DataFrame(rows)
vp = pd.concat(valid_preds, ignore_index=True)
Path("../results").mkdir(exist_ok=True)
res.to_csv("../results/glm_v2_validation.csv", index=False)
vp.to_parquet("../results/glm_v2_valid_preds.parquet", index=False)   # for comparing/blending with LightGBM

print("\nGLM v2 on the corrected target:")
print(res.pivot(index="line", columns="year", values=["D2", "A/E", "low", "top", "k"]).round(4).to_string())
print("\naverage D2 per line:", res.groupby("line").D2.mean().round(4).to_dict())

# all lines pooled, the way Kaggle scores one file
for vy in [2023, 2024]:
    v = vp[vp.year == vy]
    print(f"pooled D2 {vy}: {evaluate(v.actual, v.pred, v.weight, show=False)['D2']:.4f}")

print("\none-year-ahead A/E (actual / predicted; > 1 means the next year is under-predicted):")
print(res.pivot(index="line", columns="year", values="A/E").assign(
    mean=lambda t: t.mean(axis=1)).round(3).to_string())

motor 2023: 20s
motor 2024: 29s
buildings 2023: 11s
buildings 2024: 17s
contents 2023: 12s
contents 2024: 17s

GLM v2 on the corrected target:
               D2             A/E             low             top               k        
year         2023    2024    2023    2024    2023    2024    2023    2024    2023    2024
line                                                                                     
buildings  0.0635  0.0656  0.9917  1.0019  1.1999  0.9302  1.0743  1.0683  1.0508  1.0450
contents   0.0678  0.0653  1.0741  1.0232  1.0557  0.9670  1.1963  1.0259  1.0598  1.0563
motor      0.0503  0.0519  1.0266  1.0183  0.9505  1.0087  1.0415  1.0730  1.0213  1.0197

average D2 per line: {'buildings': 0.0645, 'contents': 0.0666, 'motor': 0.0511}
pooled D2 2023: 0.1035
pooled D2 2024: 0.1071

one-year-ahead A/E (actual / predicted; > 1 means the next year is under-predicted):
year        2023   2024   mean
line                          
buildings  0.992  1.002  0.997
contents   

In [2]:
# 08e Version C: motor one-year-ahead level factor
# Purpose: GLM v2 under-predicted the next year for motor in both folds (A/E 1.027, 1.018).
# Rule fixed before results: correct only if both folds are >= 1.01 in the same direction; factor = mean.
# Back-test added after seeing results: correcting 2024 by the 2023 factor improves motor (1.018 -> 0.991)
# but worsens contents (1.023 -> 0.953), so contents is not corrected.
MOTOR_FACTOR = (1.0266 + 1.0183) / 2

base = pd.read_csv("../submissions/glm_v2_excess_liability.csv")
is_motor = base.row_id.str.startswith("M")
assert is_motor.sum() == 254986, "motor row count changed"
sub = base.copy()
sub.loc[is_motor, "prediction"] *= MOTOR_FACTOR
assert np.isclose(sub.prediction[is_motor].sum() / base.prediction[is_motor].sum(), MOTOR_FACTOR)
assert (sub.prediction[~is_motor] == base.prediction[~is_motor]).all()
sub.to_csv("../submissions/glm_v2_motor_level.csv", index=False)
print(f"motor factor {MOTOR_FACTOR:.4f}; wrote submissions/glm_v2_motor_level.csv")

motor factor 1.0225; wrote submissions/glm_v2_motor_level.csv
